# Q1. Collecte du Dataset

Le dataset **Online Retail** a été téléchargé depuis le dépôt UCI Machine Learning Repository ou Kaggle.

Nous allons charger le fichier CSV avec Pandas afin de commencer l’analyse.

In [15]:
import pandas as pd

# Chargement du dataset
df = pd.read_csv("online_retail.csv", encoding='ISO-8859-1')

# Affichage des premières lignes
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


# Q2. Conversion CSV → Parquet

Nous sauvegardons le dataset au format Parquet afin de comparer :

- la vitesse de lecture,
- l’espace mémoire,
- les performances.

Le format Parquet est privilégié dans les systèmes Big Data car il est compressé et optimisé pour les lectures analytiques.

In [ ]:
import time

# Sauvegarde en parquet
df.to_parquet("online_retail.parquet")

# Temps lecture CSV
start = time.time()

df_csv = pd.read_csv(
    "online_retail.csv",
    encoding='ISO-8859-1'
)

csv_time = time.time() - start

# Temps lecture Parquet
start = time.time()
_
df_parquet = pd.read_parquet(
    "online_retail.parquet"
)

parquet_time = time.time() - start

print("Temps CSV :", csv_time)
print("Temps Parquet s:", parquet_time)

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

# Q3. Analyse des valeurs manquantes

Nous calculons le pourcentage de valeurs nulles dans chaque colonne.

Ensuite, nous supprimons les lignes dont `CustomerID` est manquant car cette variable est indispensable pour le clustering RFM.

In [ ]:
# Pourcentage des valeurs manquantes
missing_values = (
    df.isnull().sum() / len(df)
) * 100

print(missing_values)

# Suppression des lignes sans CustomerID
df = df.dropna(subset=['CustomerID'])

print(df.isnull().sum())

# Q4. Nettoyage métier

Nous supprimons :

- les transactions annulées (quantité négative),
- les prix unitaires nuls ou négatifs.

Cela permet de conserver uniquement les ventes réelles.

In [ ]:
# Suppression des quantités négatives
df = df[df['Quantity'] > 0]

# Suppression des prix aberrants
df = df[df['UnitPrice'] > 0]

# Vérification
print(df.describe())

# Q5. Création de la variable MontantTotal

Nous créons une nouvelle variable :

MontantTotal = Quantity × UnitPrice

Cette variable représente le montant total dépensé dans chaque transaction.

In [ ]:
# Création du montant total
df['MontantTotal'] = (
    df['Quantity'] * df['UnitPrice']
)

# Affichage
df[['Quantity', 'UnitPrice', 'MontantTotal']].head()

# Q6. Construction du DataFrame RFM

Nous transformons les données transactionnelles en données clients.

Les indicateurs RFM sont :

- Récence : nombre de jours depuis le dernier achat,
- Fréquence : nombre de factures,
- Montant : somme totale dépensée.

In [ ]:
# Conversion des dates
df['InvoiceDate'] = pd.to_datetime(
    df['InvoiceDate']
)

# Date de référence
reference_date = (
    df['InvoiceDate'].max()
    + pd.Timedelta(days=1)
)

# Création du DataFrame RFM
df_RFM = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x:
        (reference_date - x.max()).days,

    'InvoiceNo': 'nunique',

    'MontantTotal': 'sum'
}).reset_index()

# Renommer les colonnes
df_RFM.columns = [
    'CustomerID',
    'Recence',
    'Frequence',
    'Montant'
]

# Affichage
df_RFM.head()

# Q7. Analyse de la distribution du Montant

Nous traçons l’histogramme de la variable `Montant`.

Objectif :
observer la présence éventuelle d’asymétrie et de valeurs extrêmes.

In [ ]:
import matplotlib.pyplot as plt

# Histogramme
plt.figure(figsize=(8,5))

plt.hist(
    df_RFM['Montant'],
    bins=50
)

plt.title("Distribution du Montant")

plt.xlabel("Montant")

plt.ylabel("Nombre de clients")

plt.show()

# Q8. Transformation logarithmique

Nous appliquons la transformation :

log(x + 1)

Cette opération réduit l’effet des valeurs extrêmes et améliore les performances du clustering.

In [ ]:
import numpy as np

# Transformation logarithmique
df_RFM['Recence'] = np.log1p(
    df_RFM['Recence']
)

df_RFM['Frequence'] = np.log1p(
    df_RFM['Frequence']
)

df_RFM['Montant'] = np.log1p(
    df_RFM['Montant']
)

# Vérification
df_RFM.head()

# Q9. Standardisation

K-Means utilise la distance euclidienne.

Les variables doivent donc être mises sur la même échelle afin qu’aucune variable ne domine les autres.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Variables RFM
rfm_features = df_RFM[
    ['Recence', 'Frequence', 'Montant']
]

# Standardisation
scaler = StandardScaler()

RFM_scaled = scaler.fit_transform(
    rfm_features
)

print(RFM_scaled[:5])

# Q10. Méthode du coude (Elbow Method)

Nous calculons l’inertie pour plusieurs valeurs de K afin de déterminer le nombre optimal de clusters.

In [ ]:
from sklearn.cluster import KMeans

inertias = []

# Calcul des inerties
for k in range(1, 11):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42
    )

    kmeans.fit(RFM_scaled)

    inertias.append(
        kmeans.inertia_
    )

# Tracé
plt.figure(figsize=(8,5))

plt.plot(
    range(1,11),
    inertias,
    marker='o'
)

plt.xlabel("Nombre de clusters")

plt.ylabel("Inertie")

plt.title("Méthode du Coude")

plt.show()

# Q11. Validation avec le Silhouette Score

Le Silhouette Score mesure la qualité des clusters.

- proche de 1 → bon clustering,
- proche de 0 → clusters mélangés,
- négatif → mauvais clustering.

In [ ]:
from sklearn.metrics import silhouette_score

# Calcul des scores
for k in range(2, 11):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42
    )

    labels = kmeans.fit_predict(
        RFM_scaled
    )

    score = silhouette_score(
        RFM_scaled,
        labels
    )

    print(
        "K =",
        k,
        "Silhouette Score =",
        score
    )

# Q12. Modèle K-Means final

Nous entraînons le modèle final avec le nombre optimal de clusters puis nous ajoutons les labels au DataFrame RFM.

In [ ]:
# Modèle final
kmeans = KMeans(
    n_clusters=4,
    random_state=42
)

# Création des clusters
df_RFM['Cluster'] = kmeans.fit_predict(
    RFM_scaled
)

# Affichage
df_RFM.head()

# Q13. Analyse des clusters

Nous calculons les moyennes des variables RFM pour chaque cluster afin d’interpréter les profils clients.

In [ ]:
cluster_analysis = df_RFM.groupby(
    'Cluster'
)[[
    'Recence',
    'Frequence',
    'Montant'
]].mean()

print(cluster_analysis)

# Q14. Profilage des clusters

Nous identifions :

- les clients Champions :
  faible récence, fréquence élevée, montant élevé.

- les clients à risque :
  récence élevée, faible fréquence et faible montant.

In [ ]:
# Taille des clusters
print(
    df_RFM['Cluster'].value_counts()
)

# Moyennes RFM par cluster
print(cluster_analysis)

# Q15. Actions commerciales

Chaque cluster reçoit une stratégie marketing adaptée :

- Champions :
  programmes VIP et fidélisation.

- Clients fidèles :
  récompenses et promotions.

- Nouveaux clients :
  emails de bienvenue.

- Clients à risque :
  campagnes de relance et réductions.

In [ ]:
# Exemple simple d'affichage
for cluster in df_RFM['Cluster'].unique():

    print(f"\nCluster {cluster}")

    print(
        df_RFM[
            df_RFM['Cluster'] == cluster
        ][[
            'Recence',
            'Frequence',
            'Montant'
        ]].describe()
    )